# Notebook 07: Browser Tools Integration

## Learning Objectives
- Set up AgentCore Browser Tools
- Implement web scraping for attractions
- Add review aggregation logic
- Create real-time research automation
- Integrate browser tools with travel agent

## Prerequisites
- Completed Notebook 06 (Code Interpreter)
- Understanding of web scraping concepts
- Basic knowledge of HTML/CSS selectors
This notebook runs TypeScript on the Deno kernel. Pick the **Deno** kernel in the top right.


## Step 1: Browser Tools Dependencies

In [ ]:
// APPROACH A: Use credentials
// Deno.env.set("AWS_ACCESS_KEY_ID", "your_access_key");
// Deno.env.set("AWS_SECRET_ACCESS_KEY", "your_secret_key");
// Deno.env.set("AWS_SESSION_TOKEN", "your_session_token");

// APPROACH B: Use AWS SSO profile
// Deno.env.set("AWS_PROFILE", "your_profile");

Deno.env.set("AWS_REGION", "us-east-1");
console.log("\u2705 AWS configured");

In [ ]:
// Dependencies are pinned in the project's deno.json and cached by ./setup.sh, so there is no
// install step here: Deno resolves bedrock-agentcore, @strands-agents/sdk and playwright on import.
// (The Python course ran `uv add ...` in this cell.)
import { sh } from "../shared/notebook.ts";

await sh("deno", ["--version"]);

In [ ]:
import { Agent, BedrockModel } from "@strands-agents/sdk";
import { BrowserTools } from "bedrock-agentcore/experimental/browser/strands";
import { loadEnv, writeFile } from "../shared/notebook.ts";

// Load environment variables
await loadEnv();

console.log("\u2705 Strands SDK imports successful");

## Step 2: Initialize AgentCore Browser Tool

In [ ]:
// Configuration
const REGION = "us-east-1";

// Initialize the Browser tool
console.log("\ud83c\udf10 Initializing AgentCore Browser Tool...");

const browserTool = new BrowserTools({ region: REGION });

console.log(`\u2705 AgentCore Browser Tool initialized for region: ${REGION}`);
console.log(`   Tools: ${browserTool.tools.map((t) => t.name).join(", ")}`);

## Step 3: Create Travel Agent with Browser Tools

In [ ]:
// Create an agent with the Browser tool
const modelId = "us.anthropic.claude-sonnet-4-6";
const model = new BedrockModel({ modelId });

const travelAgent = new Agent({
  model,
  tools: [...browserTool.tools],
  systemPrompt: `You are an AI Travel Companion with web browsing capabilities.
    
    You can use the browser to:
    - Research attractions and points of interest
    - Find reviews and ratings for destinations
    - Gather local tips and travel information
    - Search for current travel information
    
    When users ask about destinations, use the browser to find current, accurate information.
    Provide helpful, detailed travel recommendations based on your research.`,
});

console.log("\u2705 Travel agent with browser tools created");

## Step 4: Test Browser Tools Integration

In [ ]:
// Test 1: Attraction research
console.log("\ud83e\uddea Test 1: Attraction Research");
console.log("=".repeat(50));

const prompt1 = "Goto https://www.louvre.fr/en and find out what the ticket price is.";
console.log(`User: ${prompt1}`);

const response1 = await travelAgent.invoke(prompt1);
console.log(`Agent: ${response1.toString()}`);

In [ ]:
// Test 2: Review aggregation
console.log("\ud83e\uddea Test 2: Review Research");
console.log("=".repeat(50));

const prompt2 = "I would like to go to the Paris Opera tomorrow, what are the open slots available?";
console.log(`User: ${prompt2}`);

const response2 = await travelAgent.invoke(prompt2);
console.log(`Agent: ${response2.toString()}`);

In [ ]:
// Release the remote browser session: it is billed while it lives
await browserTool.stopSession();
console.log("\ud83d\uded1 Browser session stopped");

## Step 5: Save Browser Tools Configuration

In [ ]:
// Save browser tools information for use in subsequent notebooks
const browserToolsInfo = {
  region: REGION,
  model_id: modelId,
  browser_tool_initialized: true,
  integration_status: "configured",
};

// Save to file for next notebooks
await writeFile("../backend/browser_tools_info.json", `${JSON.stringify(browserToolsInfo, null, 2)}\n`);

console.log("\ud83d\udcbe Browser tools information saved to backend/browser_tools_info.json");

## Next Steps

✅ **Completed in this notebook:**
- AgentCore Browser Tools setup with Strands SDK
- Travel agent with web browsing capabilities
- Real-time web research automation
- Natural language browser control
- Live session viewing capabilities
- Integration testing and validation